# Brent Oil Prices - Exploratory Data Analysis

This notebook performs the initial EDA for the Change Point Analysis project. We will:
1. Load and preprocess the data.
2. Visualize price trends and volatility.
3. Test for stationarity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.stattools import adfuller

plt.style.use('ggplot')

## 1. Load Data
We load the Brent Oil Prices dataset.

In [ ]:
# Try loading from different possible paths
try:
    df = pd.read_csv('../Data/BrentOilPrices.csv') # Adjusted for notebook folder structure
except FileNotFoundError:
    try:
        df = pd.read_csv('../data/raw/BrentOilPrices.csv')
    except FileNotFoundError:
        print("Error: Dataset not found.")

# Preprocessing
# Parse dates with error handling
df['Date'] = pd.to_datetime(df['Date'], format='%d-%b-%y', errors='coerce')
df.dropna(subset=['Date'], inplace=True)
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)

print(f"Data Range: {df.index.min()} to {df.index.max()}")
df.head()

## 2. Feature Engineering
Calculate Log Returns: $ r_t = \ln(P_t / P_{t-1}) $

In [ ]:
df['Log_Returns'] = np.log(df['Price'] / df['Price'].shift(1))
df.dropna(inplace=True)

## 3. Visualization

In [ ]:
# Price History
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Price'], label='Brent Price')
plt.title('Brent Oil Prices (1987-2022)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

In [ ]:
# Volatility (Log Returns)
plt.figure(figsize=(12, 6))
plt.plot(df.index, df['Log_Returns'], label='Log Returns', alpha=0.7)
plt.title('Brent Oil Price Log Returns (Volatility)')
plt.xlabel('Date')
plt.show()

## 4. Stationarity Tests (ADF)
Augmented Dickey-Fuller test to check for unit roots.

In [ ]:
def adf_check(timeseries, name):
    result = adfuller(timeseries, autolag='AIC')
    print(f"ADF Test on {name}:")
    print(f"Test Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    if result[1] <= 0.05:
        print("Result: Stationary (Reject H0)")
    else:
        print("Result: Non-Stationary (Fail to reject H0)")
    print("-" * 30)

adf_check(df['Price'], "Price")
adf_check(df['Log_Returns'], "Log Returns")